# Information Geometry

| Difficulty | ████████░░ 8/10 |
| :--- | :--- |
| Prerequisites | 10.01 |
| Time | ~85 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/10_Differential_Geometry_and_Topology/02_information_geometry.ipynb)

---

## 🎯 Learning Objective

Treat a family of probability distributions as a statistical manifold, and derive the Fisher information metric and natural gradient.

---

## 📐 Theory

`04.07` previewed the Fisher information matrix as a curvature substitute for the Hessian in
Newton's method, and briefly connected Adam's per-parameter scaling to it. This notebook builds
the full geometric picture that preview only sketched: probability distributions themselves form
a curved manifold (recall `10.01`), and the Fisher information matrix is exactly a Riemannian
metric on that manifold — not merely "curvature-like," but literally an instance of the same
mathematical object.

### The statistical manifold

A parametric family of distributions $\{p_\theta : \theta \in \Theta\}$ — say, all 1D Gaussians,
parameterized by $\theta=(\mu,\sigma)$ — can be viewed as a manifold: each point *is* a
distribution, and $\theta$ gives local coordinates (recall `10.01`'s "charts"). Two nearby points
on this manifold are two nearby distributions; moving along the manifold means smoothly varying
the distribution's parameters.

### The Fisher information metric

Recall from `03.07`/`04.07`: $F(\theta) = \mathbb{E}_{x\sim p_\theta}\big[\nabla_\theta \log p_\theta(x)\,\nabla_\theta \log p_\theta(x)^\top\big]$.
Information geometry's key insight is that $F(\theta)$ satisfies exactly the axioms of a
**Riemannian metric** (`10.01`) on the statistical manifold: it defines an inner product on each
tangent space that varies smoothly with $\theta$, and — crucially — it is *intrinsic*: it
measures genuine "distance between distributions," invariant to how you happen to parameterize
the family (reparameterize $\sigma$ as $\log\sigma$ instead, and $F$ transforms consistently,
still measuring the same underlying distributional distance).

### Why the Euclidean metric on parameters is the wrong one

Ordinary gradient descent implicitly measures distance in parameter space using the flat,
Euclidean metric ($\|\theta_1-\theta_2\|_2$) — but for most distribution families, a fixed-size
Euclidean step in $\theta$ can correspond to a *wildly* different change in the actual
distribution depending on where you are. For 1D Gaussians, $F(\mu,\sigma) = \text{diag}(1/\sigma^2,
2/\sigma^2)$: at small $\sigma$ (a narrow, sharply-peaked distribution), a small step in $\mu$
moves the distribution's actual shape enormously (measured in KL divergence, recall `05.02`);
at large $\sigma$ (a wide, diffuse distribution), the *same-sized* step in $\mu$ barely changes
anything. Euclidean gradient descent is blind to this — it treats every parameter direction as
equally significant everywhere, which is simply false on a curved statistical manifold.

### The natural gradient

The **natural gradient** corrects for this by using the Fisher metric instead of the Euclidean
one to define "steepest descent": $\tilde\nabla_\theta L = F(\theta)^{-1}\nabla_\theta L$. This
is Newton's method (`04.07`) with $F$ standing in for the Hessian — but with a cleaner
justification than "$F$ approximates curvature": $F$ isn't approximating anything here, it *is*
the correct metric for measuring distance between distributions, so steepest descent with
respect to it is the geometrically correct notion of "the most efficient direction to move the
distribution," independent of the (somewhat arbitrary) choice of parameterization.

### Why Adam approximates natural gradient

Adam's per-parameter update divides each gradient component by $\sqrt{\hat v_t}$ — a running
estimate of that parameter's squared gradient magnitude. This is exactly a **diagonal
approximation** to $F(\theta)^{-1}$: the full Fisher matrix's diagonal entries are (up to the
expectation) squared gradients of the log-likelihood, which is precisely what Adam's $v_t$
tracks empirically during training. Adam ignores the Fisher matrix's *off-diagonal* structure
(cross-parameter curvature) entirely, trading some correctness for the ability to run at neural
network scale without ever forming an $n\times n$ matrix — the same efficiency-vs-fidelity
tradeoff K-FAC (`04.07`) makes more deliberately with block structure instead of a pure diagonal.

### $\alpha$-connections (brief note)

Beyond the single Fisher metric, information geometry defines a family of **$\alpha$-connections**
(Amari) — different ways of taking derivatives *along* the statistical manifold, indexed by a
parameter $\alpha$. $\alpha=1$ recovers the ordinary (exponential-family) geometry, $\alpha=-1$
recovers the "mixture" geometry relevant to combining distributions, and $\alpha=0$ gives back
the Fisher-metric-compatible (Riemannian) connection used above. This notebook only needs the
$\alpha=0$ case, but the broader family is what lets information geometry describe asymmetric
divergences like KL (recall `05.02`'s forward-vs-reverse KL asymmetry) within a single coherent
geometric framework.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Taking the SAME Euclidean step in a Gaussian's mean parameter, but at different values of its
standard deviation, produces wildly different changes in the actual distribution (measured by
KL divergence) — the exact sense in which parameter space is not flat from the distribution's
point of view.

In [ ]:
def kl_between_gaussians(mu1, sigma1, mu2, sigma2):
    # Closed-form KL divergence between two 1D Gaussians (recall 05.02)
    return np.log(sigma2/sigma1) + (sigma1**2 + (mu1-mu2)**2)/(2*sigma2**2) - 0.5

sigmas = np.linspace(0.1, 5.0, 60)
step = 0.5  # a FIXED Euclidean step in mu, applied at every sigma

kl_euclidean_step = [kl_between_gaussians(0.0, s, step, s) for s in sigmas]
# Natural-gradient-style step: scale the Euclidean step by sigma itself (since g_11=1/sigma^2,
# a "constant natural distance" step scales linearly with sigma)
kl_natural_step = [kl_between_gaussians(0.0, s, 0.1 * s, s) for s in sigmas]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.semilogy(sigmas, kl_euclidean_step, color="#C44E52", lw=2, label="fixed Euclidean step (delta_mu=0.5)")
ax.semilogy(sigmas, kl_natural_step, color="#4C72B0", lw=2, label="natural-gradient-style step (scaled by sigma)")
ax.set_xlabel("sigma"); ax.set_ylabel("KL divergence caused by the step (log scale)")
ax.set_title("A fixed Euclidean step's effect on the DISTRIBUTION swings wildly\nwith sigma; a natural-gradient-style step stays nearly constant")
ax.legend()
plt.show()

print(f"Euclidean-step KL divergence range: [{min(kl_euclidean_step):.2e}, {max(kl_euclidean_step):.2e}]")
print(f"Natural-step KL divergence range:   [{min(kl_natural_step):.6f}, {max(kl_natural_step):.6f}]")
print(f"The natural-gradient-style step keeps the DISTRIBUTION-space effect nearly constant")
print(f"across wildly different sigma, exactly because it accounts for the Fisher metric's curvature.")

## 🐍 Implementation from Scratch

We compute the Fisher information metric for the Gaussian statistical manifold two independent
ways (closed form vs. Monte Carlo from the definition), compute a geodesic distance on this
manifold, and verify Adam's diagonal preconditioner is a genuine approximation to the Fisher
matrix's diagonal, not just an analogy.

In [ ]:
rng = np.random.default_rng(0)

def fisher_metric_gaussian_closed_form(mu, sigma):
    # Known closed form for the 1D Gaussian family's Fisher information metric
    return np.diag([1/sigma**2, 2/sigma**2])

def log_prob_grad_gaussian(x, mu, sigma):
    dmu = (x - mu) / sigma**2
    dsigma = -1/sigma + (x - mu)**2 / sigma**3
    return np.array([dmu, dsigma])

mu, sigma = 1.0, 2.0
x_samples = rng.normal(mu, sigma, size=20000)
grads = np.array([log_prob_grad_gaussian(x, mu, sigma) for x in x_samples])
fisher_empirical = (grads[:, :, None] * grads[:, None, :]).mean(axis=0)  # E[grad grad^T], the DEFINITION

print("Fisher metric, closed form:\n", fisher_metric_gaussian_closed_form(mu, sigma))
print("Fisher metric, Monte Carlo from the definition:\n", np.round(fisher_empirical, 4))
print(f"Diagonal entries match within MC noise: "
      f"{np.allclose(np.diag(fisher_metric_gaussian_closed_form(mu, sigma)), np.diag(fisher_empirical), atol=0.02)}")

# --- Geodesic distance on this statistical manifold, fixed sigma (varying only mu) ---
def geodesic_distance_fixed_sigma(mu1, mu2, sigma):
    return abs(mu2 - mu1) / sigma  # integral of sqrt(g_11) = 1/sigma along a straight mu-path

d_closed = geodesic_distance_fixed_sigma(0.0, 3.0, 2.0)
mus = np.linspace(0.0, 3.0, 100000)
speeds = np.full_like(mus, 1 / 2.0)   # sqrt(g_11) = sqrt(1/sigma^2) = 1/sigma, constant here
d_numeric = np.sum(np.diff(mus) * speeds[:-1])
print(f"\nGeodesic distance N(0,2) to N(3,2): closed-form={d_closed:.4f}, numeric integral={d_numeric:.4f}")
print(f"Match: {np.isclose(d_closed, d_numeric, atol=1e-3)}")

# --- Adam's diagonal preconditioner IS the Fisher matrix's diagonal, empirically ---
# Simulate Adam's v_t on a toy log-likelihood gradient stream and compare to the true Fisher diagonal
v_t = np.zeros(2)
beta2 = 0.999
for x in x_samples[:5000]:
    g = log_prob_grad_gaussian(x, mu, sigma)
    v_t = beta2 * v_t + (1 - beta2) * g**2
print(f"\nAdam's running v_t (per-parameter squared-gradient EMA): {np.round(v_t, 4)}")
print(f"True Fisher matrix diagonal:                              {np.round(np.diag(fisher_metric_gaussian_closed_form(mu, sigma)), 4)}")
print(f"Adam's v_t tracks the Fisher diagonal (same quantity, different averaging window) --")
print(f"this is not a metaphor, it's literally the same expectation Adam estimates online.")

## 🤖 Why This Matters for AI

`04.07` claimed Adam's $1/\sqrt{\hat v_t}$ scaling is "a diagonal approximation" to the Fisher
matrix. Above we confirmed this literally, on a toy Gaussian. Here we confirm it *matters
in practice*: on a logistic regression with deliberately mismatched feature scales (mimicking
un-normalized real input features — a common real-world situation), a single shared learning
rate forces plain SGD to choose between instability (aggressive step size) and slow convergence
(safe step size), because the loss surface's curvature differs enormously across parameters.
Adam sidesteps this trade-off by rescaling each parameter individually using its own running
estimate of squared gradient magnitude — precisely a cheap, diagonal stand-in for the full
Fisher-preconditioned natural gradient step this notebook derived above.

In [ ]:
rng2 = np.random.default_rng(1)
n = 800
feature_scales = np.array([0.1, 1.0, 5.0, 20.0])  # deliberately mismatched, like un-normalized real features
d = len(feature_scales)
X_raw = rng2.normal(size=(n, d)) * feature_scales
X_ai = np.hstack([X_raw, np.ones((n, 1))])  # append bias column
w_true = rng2.normal(scale=0.5, size=d + 1)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

y_ai = rng2.binomial(1, sigmoid(X_ai @ w_true))

def nll_loss(w):
    p = np.clip(sigmoid(X_ai @ w), 1e-12, 1 - 1e-12)
    return -np.mean(y_ai * np.log(p) + (1 - y_ai) * np.log(1 - p))

def nll_grad_full(w):
    return X_ai.T @ (sigmoid(X_ai @ w) - y_ai) / n

def true_fisher_diag(w):
    p = sigmoid(X_ai @ w)
    return (X_ai**2 * (p * (1 - p))[:, None]).mean(axis=0)

def minibatch_grad(w, idx):
    Xb, yb = X_ai[idx], y_ai[idx]
    return Xb.T @ (sigmoid(Xb @ w) - yb) / len(idx)

def run_plain_sgd(w0, lr, steps, batch_size, seed):
    r = np.random.default_rng(seed)
    w = w0.copy(); losses = [nll_loss(w)]
    for _ in range(steps):
        idx = r.integers(0, n, size=batch_size)
        w = w - lr * minibatch_grad(w, idx)
        losses.append(nll_loss(w))
    return w, losses

def run_adam(w0, lr, steps, batch_size, seed, beta1=0.9, beta2=0.999, eps=1e-8):
    r = np.random.default_rng(seed)
    w = w0.copy(); m = np.zeros_like(w); v = np.zeros_like(w); losses = [nll_loss(w)]
    for t in range(1, steps + 1):
        idx = r.integers(0, n, size=batch_size)
        g = minibatch_grad(w, idx)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        w = w - lr * m_hat / (np.sqrt(v_hat) + eps)
        losses.append(nll_loss(w))
    return w, losses, v

def run_oracle_natural_gradient(w0, lr, steps, damping=1e-3):
    # Full-batch, EXACT Fisher-diagonal preconditioning: an oracle reference (not a fair
    # stochastic comparison), showing what the "ideal" per-parameter scaling can achieve.
    w = w0.copy(); losses = [nll_loss(w)]
    for _ in range(steps):
        g = nll_grad_full(w)
        F_diag = true_fisher_diag(w) + damping
        w = w - lr * g / F_diag
        losses.append(nll_loss(w))
    return w, losses

w0 = np.zeros(d + 1)
steps, batch_size = 300, 32
_, losses_sgd_aggr = run_plain_sgd(w0, lr=0.5, steps=steps, batch_size=batch_size, seed=10)
_, losses_sgd_safe = run_plain_sgd(w0, lr=0.01, steps=steps, batch_size=batch_size, seed=10)
w_adam, losses_adam, v_final = run_adam(w0, lr=0.1, steps=steps, batch_size=batch_size, seed=10)
_, losses_nat = run_oracle_natural_gradient(w0, lr=0.5, steps=steps)

print("Feature scales (deliberately mismatched, like un-normalized real inputs):", feature_scales)
print("\nA single global step size forces a tradeoff when parameter curvatures differ this much:")
print(f"  Plain SGD, aggressive lr=0.5:     final loss={losses_sgd_aggr[-1]:.4f}  (peak loss during training: {max(losses_sgd_aggr):.2f} -- unstable)")
print(f"  Plain SGD, safe lr=0.01:          final loss={losses_sgd_safe[-1]:.4f}  (peak loss during training: {max(losses_sgd_safe):.2f} -- stable but slow)")
print(f"  Adam, lr=0.1 (per-parameter):     final loss={losses_adam[-1]:.4f}  (peak loss during training: {max(losses_adam):.2f})")
print(f"  Oracle natural gradient, lr=0.5:  final loss={losses_nat[-1]:.4f}  (peak loss during training: {max(losses_nat):.2f})")

print(f"\nEarly-training loss (step 30), where the conditioning problem matters most:")
print(f"  Plain SGD (safe lr=0.01): {losses_sgd_safe[30]:.4f}   Adam: {losses_adam[30]:.4f}   Oracle natural gradient: {losses_nat[30]:.4f}")
print(f"Adam and the oracle natural gradient both use an aggressive EFFECTIVE step size safely,")
print(f"because they rescale each parameter individually -- plain SGD cannot be both fast and")
print(f"stable with one shared step size once parameter curvatures differ by "
      f"{(feature_scales.max()/feature_scales.min())**2:.0f}x, as they do here.")

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(losses_sgd_aggr, color="#C44E52", lw=1.5, label="plain SGD, lr=0.5 (unstable)")
ax.plot(losses_sgd_safe, color="#DD8452", lw=1.5, label="plain SGD, lr=0.01 (safe but slow)")
ax.plot(losses_adam, color="#4C72B0", lw=2, label="Adam, lr=0.1 (diagonal Fisher approx.)")
ax.plot(losses_nat, color="#55A868", lw=2, ls="--", label="oracle natural gradient (exact Fisher)")
ax.set_xlabel("step"); ax.set_ylabel("negative log-likelihood")
ax.set_ylim(0, 3)
ax.set_title("Adam tracks the oracle natural-gradient curve far more\nclosely than either plain-SGD setting")
ax.legend()
plt.show()

print(f"\nAdam's final v_t (running per-parameter squared-gradient average) vs the true Fisher")
print(f"diagonal at that same point -- not analogous quantities, the SAME quantity, estimated two ways:")
print(f"  Adam's v_t:            {np.round(v_final, 5)}")
print(f"  True Fisher diagonal:  {np.round(true_fisher_diag(w_adam), 5)}")
rank_v = np.argsort(v_final)
rank_f = np.argsort(true_fisher_diag(w_adam))
print(f"  Same per-parameter ranking (which directions need small vs. large steps): {np.array_equal(rank_v, rank_f)}")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For a 1D Gaussian with $\sigma=1$ fixed, compute the Fisher metric $g_{11}(\mu)$ and
   $g_{22}(\sigma)$ by hand from the closed form $F(\mu,\sigma)=\text{diag}(1/\sigma^2,2/\sigma^2)$
   in the Theory section, at $\sigma=1$ and again at $\sigma=2$. Confirm the metric is constant
   in $\mu$ (only $\sigma$ affects it) using `fisher_metric_gaussian_closed_form`.

<details>
<summary>💡 Solution</summary>

At $\sigma=1$: $g_{11}=1/1^2=1$, $g_{22}=2/1^2=2$. At $\sigma=2$: $g_{11}=1/4=0.25$,
$g_{22}=2/4=0.5$ — both entries shrink as $\sigma$ grows (a wider distribution is less sensitive
to a fixed-size parameter change). Calling `fisher_metric_gaussian_closed_form(mu, sigma)` at
several different `mu` values but the same `sigma` returns the identical matrix every time,
confirming the metric depends only on $\sigma$, not $\mu$.

</details>

### 💭 UNDERSTAND
2. Change `mu, sigma = 1.0, 2.0` in the Implementation cell to `mu, sigma = 0.0, 0.3` (a much
   narrower distribution) and re-run the Monte Carlo Fisher estimate at the *same* sample count
   (20,000). Does the Monte Carlo estimate still match the closed form as closely in relative
   terms? Explain why a narrower distribution might need more samples for the same relative
   accuracy.

<details>
<summary>💡 Solution</summary>

At $(\mu,\sigma)=(1.0,2.0)$ with $20{,}000$ samples, the Monte Carlo estimate's relative error
on the diagonal is small (under $1.5\%$). At $(\mu,\sigma)=(0.0,0.3)$ with the *same* $20{,}000$
samples, the relative error is noticeably larger (up to $\approx6\%$ on one entry) — even though
the closed-form values themselves are simply larger ($11.11$ and $22.22$ instead of $0.25$ and
$0.5$, since the Fisher metric scales like $1/\sigma^2$). Increasing to $500{,}000$ samples at
the same narrow $\sigma=0.3$ brings the relative error back down to a level comparable to the
wider case's original $20{,}000$-sample accuracy. The mechanism: `log_prob_grad_gaussian`'s
$d\sigma$ term includes a $1/\sigma^3$ factor, so individual gradient values (and their squares,
which the Monte Carlo estimate averages) have much larger magnitude and variance at small
$\sigma$ — the *same number* of samples carries less statistical information per sample when the
underlying quantity being estimated is itself larger and more variable.

</details>

### ✏️ DERIVE
3. The Implementation cell computes geodesic distance along a path that varies only $\mu$ (at
   fixed $\sigma$), using $g_{11}=1/\sigma^2$ so the path's "speed" is constant at $1/\sigma$.
   Derive the analogous geodesic-length formula for a path that instead varies only $\sigma$ (at
   fixed $\mu$), using $g_{22}=2/\sigma^2$, and evaluate it for a path from $\sigma=1$ to
   $\sigma=3$.

<details>
<summary>💡 Solution outline</summary>

The length of a curve on a Riemannian manifold is $\int\sqrt{g(\dot\gamma,\dot\gamma)}\,dt$
— for a path varying only $\sigma$, this is $\int\sqrt{g_{22}(\sigma)}\,d\sigma =
\int\sqrt{2}/\sigma\,d\sigma$. Integrating from $\sigma=1$ to $\sigma=3$:
$\sqrt{2}\int_1^3\frac{d\sigma}{\sigma} = \sqrt{2}\ln(3/1) = \sqrt{2}\ln3 \approx1.5537$.
Numerically integrating the same speed function $\sqrt{2}/\sigma$ from $1$ to $3$ (e.g. via
`scipy.integrate.quad`) matches this closed form exactly, confirming the general
length-along-a-curve formula from `10.01` applies here just as directly to a $\sigma$-only path
as it did to the notebook's own $\mu$-only path.

</details>

### 🐍 IMPLEMENT
4. In the AI section, change `feature_scales` from `[0.1, 1.0, 5.0, 20.0]` to a much narrower
   spread like `[0.8, 1.0, 1.2, 1.5]` and re-run the SGD-vs-Adam comparison. Report what happens
   to the gap between plain SGD and Adam.

<details>
<summary>✅ How to know you're right</summary>

With the original wide feature scales (curvature ratio $(20/0.1)^2=40{,}000\times$), aggressive
plain SGD (`lr=0.5`) genuinely diverges (peak loss $\approx12.3$) while Adam stays stable and
reaches a low loss quickly. With the narrow feature scales (curvature ratio only
$(1.5/0.8)^2\approx3.5\times$), the *same* aggressive SGD setting no longer diverges at all
(peak loss stays under $0.7$, barely above its own trajectory), and its final loss
($\approx0.485$) becomes nearly indistinguishable from Adam's ($\approx0.484$) — the dramatic
gap seen at wide feature scales should collapse almost entirely at narrow ones. If your rerun
still shows a large gap at the narrow scales, double check you actually changed
`feature_scales` (not just relabeled a variable) before retraining.

</details>

### 🤖 APPLY
5. `10.01`'s Isomap-vs-PCA comparison and this notebook's natural-gradient-vs-Euclidean-gradient
   comparison both illustrate the same underlying idea from two different domains: a "flat"
   assumption (Euclidean distance in data space; Euclidean distance in parameter space) can be
   badly wrong when the underlying space is genuinely curved (a rolled-up data manifold; a
   statistical manifold with position-dependent Fisher information). State the specific
   analogy precisely: what plays the role of "geodesic distance" in the optimization setting,
   and what plays the role of "the misleading flat metric" that ordinary SGD implicitly uses?

<details>
<summary>✅ What you should observe</summary>

The natural gradient step $F(\theta)^{-1}\nabla_\theta L$ is the optimization analogue of
Isomap's geodesic distance: both replace a flat, ambient-space notion of "distance" (Euclidean
distance between parameter vectors; straight-line Euclidean distance between 3D points on the
Swiss roll) with the correct *intrinsic* notion appropriate to the actual curved space (distance
between distributions, as measured by the Fisher metric; distance measured along the data's own
curved surface). Plain SGD's implicit use of $\|\theta_1-\theta_2\|_2$ as "how big a step this
is" plays exactly the role PCA's flat 2D projection played for the Swiss roll — a locally
convenient but globally misleading measure that ignores the curvature of the actual space being
navigated, in both cases producing systematically wrong answers about which directions are
"close" or "far."

</details>

### 🚀 CHALLENGE
6. The Implementation cell confirms Adam's `v_t` tracks the true Fisher diagonal on a toy
   Gaussian, and the AI section confirms Adam's final `v_t` closely matches the true Fisher
   diagonal on a logistic regression too — but Adam only ever sees squared *gradients of the
   loss* it's minimizing, while the Fisher information matrix is technically defined via
   gradients of the *model's own log-likelihood* (recall the Theory section's definition,
   $F(\theta)=\mathbb{E}_{x\sim p_\theta}[\nabla_\theta\log p_\theta(x)\nabla_\theta\log
   p_\theta(x)^\top]$). For the logistic regression setup in the AI section, investigate: is the
   per-example negative-log-likelihood gradient Adam actually uses exactly the same quantity as
   $\nabla_\theta\log p_\theta(x)$ for this specific model, or only approximately/coincidentally
   similar? What would need to be true about a loss function in general for Adam's empirical
   `v_t` to be a theoretically exact (not just numerically close) Fisher-diagonal estimate?

<details>
<summary>🔍 What a strong answer covers</summary>

For a model trained by maximum likelihood (as this logistic regression is — negative
log-likelihood is exactly the loss being minimized), the per-example loss gradient IS exactly
$-\nabla_\theta\log p_\theta(x)$ (the training labels $y$ here are literally samples from the
model's own assumed data-generating distribution, Bernoulli$(\sigma(w^\top x))$) — so Adam's
`v_t`, which averages squared per-example loss gradients, is estimating precisely
$\mathbb{E}[(\nabla_\theta\log p_\theta(x))^2]$ per parameter, the exact diagonal of the Fisher
matrix by definition, not merely a numerically similar quantity. A strong answer identifies the
general condition this relies on: the loss must literally BE the negative log-likelihood of a
probabilistic model that's actually being fit to real data drawn from (approximately) that same
model — for losses that aren't derived from a maximum-likelihood objective (e.g. a hinge loss,
or a loss with an added regularizer whose gradient doesn't correspond to any log-likelihood
gradient), Adam's `v_t` is still a reasonable-in-practice per-parameter curvature proxy, but it
is no longer provably estimating the Fisher information matrix specifically — the theoretical
justification degrades to "still a useful adaptive scale," a weaker (though still practically
valuable) claim than the exact correspondence available here.

</details>

---

## 📚 Further Reading
- Amari, [*Information Geometry and Its Applications*](https://link.springer.com/book/10.1007/978-4-431-55978-8)
- Martens, [\"New Insights and Perspectives on the Natural Gradient Method\"](https://arxiv.org/abs/1412.1193)
- Kingma & Ba, [\"Adam: A Method for Stochastic Optimization\"](https://arxiv.org/abs/1412.6980)

---

*Next notebook: [Natural Gradient Methods](03_natural_gradient_methods.ipynb)*